In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.types import *

In [0]:
initial_path = "/Volumes/final_project/source/raw_data/loan_repayment_initial.csv"

print("Initial file:", initial_path)
 

In [0]:
display(dbutils.fs.ls("/Volumes/final_project/source/raw_data/loan_repayment_initial.csv"))

In [0]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/final_project/source/raw_data/loan_repayment_initial.csv")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
print("Total source records:", df.count())

In [0]:
from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
]).display()

In [0]:
bronze_df = df.withColumn(
    "source_file", col("_metadata.file_path")
).withColumn(
    "batch_id", lit("BATCH_001")
).withColumn(
    "ingestion_timestamp", current_timestamp()
)


In [0]:
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_project.bronze.loan_repayment")

In [0]:
print("Bronze records:", bronze_df.count())

In [0]:
validated_df = bronze_df.withColumn(
    "rejection_reason",
    concat_ws(", ",
        when(col("payment_id").isNull(), "Missing payment ID"),

        when(col("payment_date").isNull(), "Missing payment date"),

        when(col("loan_id").isNull(), "Missing loan ID"),

        when(col("customer_id").isNull(), "Missing customer ID"),

        when(
            col("loan_type").isNull() |
            (~upper(trim(col("loan_type"))).isin("PERSONAL", "HOME", "VEHICLE")),
            "Invalid loan type"
        ),

        when(
            col("branch_city").isNull() |
            (trim(col("branch_city")) == ""),
            "Missing branch city"
        ),

        when(
            col("principal_amount").isNull() |
            (col("principal_amount") <= 0),
            "Invalid principal amount"
        ),

        when(
            col("interest_amount").isNull() |
            (col("interest_amount") <= 0),
            "Invalid interest amount"
        ),

        when(
            col("penalty_amount").isNull() |
            (col("penalty_amount") < 0),
            "Invalid penalty amount"
        ),

        when(
            col("payment_status").isNull() |
            (~upper(trim(col("payment_status"))).isin("PAID", "PENDING", "FAILED")),
            "Invalid payment status"
        )
    )
)

In [0]:
duplicate_ids = bronze_df.groupBy("payment_id") \
    .count() \
    .filter(
        (col("count") > 1) & col("payment_id").isNotNull()
    )

display(duplicate_ids)

In [0]:
duplicate_payment_ids = duplicate_ids.select(
    "payment_id"
).distinct().withColumn(
    "duplicate_flag", lit(1)
)

validated_df = validated_df.join(
    duplicate_payment_ids,
    on="payment_id",
    how="left"
)

validated_df = validated_df.withColumn(
    "rejection_reason",
    when(
        col("duplicate_flag") == 1,
        concat_ws(
            ", ",
            col("rejection_reason"),
            lit("Duplicate payment ID")
        )
    ).otherwise(col("rejection_reason"))
).drop("duplicate_flag")

display(validated_df)

In [0]:
valid_df = validated_df.filter(
    col("rejection_reason") == ""
)

rejected_df = validated_df.filter(
    col("rejection_reason") != ""
)

print("Valid records:", valid_df.count())
print("Rejected records:", rejected_df.count())

display(valid_df)
display(rejected_df)

In [0]:
bronze_count = bronze_df.count()
valid_count = valid_df.count()
rejected_count = rejected_df.count()

print("Bronze records:", bronze_count)
print("Valid records:", valid_count)
print("Rejected records:", rejected_count)

if bronze_count == valid_count + rejected_count:
    print("Reconciliation: PASS")
else:
    print("Reconciliation: FAIL")